In [ ]:
!wget https://raw.githubusercontent.com/Pirogjikdhima/Diploma/refs/heads/main/Corpus/Combined/combined_dataset.csv

--2025-09-18 07:52:56--  https://raw.githubusercontent.com/Pirogjikdhima/Diploma/refs/heads/main/Corpus/Combined/combined_dataset.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26130198 (25M) [text/plain]
Saving to: ‘combined_dataset.csv’

combined_dataset.cs 100%[===================>]  24.92M   139MB/s    in 0.2s    

2025-09-18 07:52:57 (139 MB/s) - ‘combined_dataset.csv’ saved [26130198/26130198]



In [ ]:
!pip install seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=6230f1e77c5a796546ad4b742ae4590e46431164137a449d2d0724befeac59d1
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
import time
import numpy as np
import pandas as pd
import xgboost as xgb
from seqeval.metrics import f1_score as entity_f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from sklearn.neural_network import MLPClassifier

In [ ]:
dataset = pd.read_csv("combined_dataset.csv")

In [ ]:
class MachineLearningAlgorithmsNER:
    def __init__(self, model_name, dataset):
        self.model_name = model_name
        self.dataset = dataset

        self.X = None
        self.Y = None

        self.word2idx = None
        self.idx2word = None
        self.tag2idx = None
        self.idx2tag = None

        self.model = None

    def get_word_and_tags_2_ids(self):
        words = self.dataset['WORD'].values
        ner_tags = self.dataset['NER_TAG'].values

        word_encoder = LabelEncoder()
        tag_encoder = LabelEncoder()

        self.X = word_encoder.fit_transform(words).reshape(-1, 1)
        self.Y = tag_encoder.fit_transform(ner_tags)

        self.word2idx = {word: idx for idx, word in enumerate(word_encoder.classes_)}
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}

        self.tag2idx = {tag: idx for idx, tag in enumerate(tag_encoder.classes_)}
        self.idx2tag = {idx: tag for tag, idx in self.tag2idx.items()}

    def split_dataset(self, test_size=0.2, random_state=42):
        X_train, X_test, Y_train, Y_test = train_test_split(
            self.X, self.Y, test_size=test_size, random_state=random_state, stratify=self.Y
        )
        return X_train, X_test, Y_train, Y_test

    def _train_and_evaluate(self, model, X_train, y_train, X_test, y_test):
        start_time = time.time()
        model.fit(X_train, y_train)
        training_time = time.time() - start_time

        y_pred = model.predict(X_test)

        token_f1 = f1_score(y_test, y_pred, average='micro')

        y_true_tags = [self.idx2tag[idx] for idx in y_test]
        y_pred_tags = [self.idx2tag[idx] for idx in y_pred]

        entity_f1 = entity_f1_score([y_true_tags], [y_pred_tags])

        self.model = model

        print(f"Finished Training: {self.model_name}")

        return {
            'Model': self.model_name,
            'Token F1': token_f1,
            'Entity F1': entity_f1,
            'Training Time': training_time
        }

    def train_naive_bayes(self, X_train, y_train, X_test, y_test):
        return self._train_and_evaluate(MultinomialNB(), X_train, y_train, X_test, y_test)

    def train_random_forest(self, X_train, y_train, X_test, y_test):
        return self._train_and_evaluate(
            RandomForestClassifier(n_jobs=-1),
            X_train, y_train, X_test, y_test
        )

    def train_extra_trees(self, X_train, y_train, X_test, y_test):
        return self._train_and_evaluate(
            ExtraTreesClassifier(n_jobs=-1),
            X_train, y_train, X_test, y_test
        )

    def train_gradient_boosting(self, X_train, y_train, X_test, y_test):
        return self._train_and_evaluate(
            GradientBoostingClassifier(),
            X_train, y_train, X_test, y_test
        )

    def train_xgboost(self, X_train, y_train, X_test, y_test):
        return self._train_and_evaluate(
            xgb.XGBClassifier(n_estimators=100, eval_metric='mlogloss'),
            X_train, y_train, X_test, y_test
        )


def naive_bayes(model_name, dataset):
    model = MachineLearningAlgorithmsNER(model_name, dataset)
    model.get_word_and_tags_2_ids()
    X_train, X_test, Y_train, Y_test = model.split_dataset()
    result = model.train_naive_bayes(X_train, Y_train, X_test, Y_test)
    return result

def extra_trees(model_name, dataset):
    model = MachineLearningAlgorithmsNER(model_name, dataset)
    model.get_word_and_tags_2_ids()
    X_train, X_test, Y_train, Y_test = model.split_dataset()
    result = model.train_extra_trees(X_train, Y_train, X_test, Y_test)
    return result

def random_forest(model_name, dataset):
    model = MachineLearningAlgorithmsNER(model_name, dataset)
    model.get_word_and_tags_2_ids()
    X_train, X_test, Y_train, Y_test = model.split_dataset()
    result = model.train_random_forest(X_train, Y_train, X_test, Y_test)
    return result

def xgboost(model_name, dataset):
    model = MachineLearningAlgorithmsNER(model_name, dataset)
    model.get_word_and_tags_2_ids()
    X_train, X_test, Y_train, Y_test = model.split_dataset()
    result = model.train_xgboost(X_train, Y_train, X_test, Y_test)
    return result

def gradient_boosting(model_name, dataset):
    model = MachineLearningAlgorithmsNER(model_name, dataset)
    model.get_word_and_tags_2_ids()
    X_train, X_test, Y_train, Y_test = model.split_dataset()
    result = model.train_gradient_boosting(X_train, Y_train, X_test, Y_test)
    return result

In [ ]:
class MachineLearningAlgorithmsNERFeatures:
    def __init__(self, model_name, dataset):
        self.model_name = model_name
        self.dataset = dataset

        self.X = None
        self.Y = None

        self.word2idx = None
        self.idx2word = None
        self.pos2idx = None
        self.idx2pos = None
        self.lemma2idx = None
        self.idx2lemma = None
        self.tag2idx = None
        self.idx2tag = None

        self.word_encoder = None
        self.pos_encoder = None
        self.lemma_encoder = None
        self.tag_encoder = None

        self.model = None

    def get_word_and_tags_2_ids(self):
        words = self.dataset['WORD'].values
        pos_tags = self.dataset['POS_TAG'].values
        lemmas = self.dataset['LEMMA'].values
        ner_tags = self.dataset['NER_TAG'].values

        self.word_encoder = LabelEncoder()
        self.pos_encoder = LabelEncoder()
        self.lemma_encoder = LabelEncoder()
        self.tag_encoder = LabelEncoder()

        word_encoded = self.word_encoder.fit_transform(words).reshape(-1, 1)
        pos_encoded = self.pos_encoder.fit_transform(pos_tags).reshape(-1, 1)
        lemma_encoded = self.lemma_encoder.fit_transform(lemmas).reshape(-1, 1)

        self.X = np.hstack([word_encoded, pos_encoded, lemma_encoded])
        self.Y = self.tag_encoder.fit_transform(ner_tags)

        self.word2idx = {word: idx for idx, word in enumerate(self.word_encoder.classes_)}
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}

        self.pos2idx = {pos: idx for idx, pos in enumerate(self.pos_encoder.classes_)}
        self.idx2pos = {idx: pos for pos, idx in self.pos2idx.items()}

        self.lemma2idx = {lemma: idx for idx, lemma in enumerate(self.lemma_encoder.classes_)}
        self.idx2lemma = {idx: lemma for lemma, idx in self.lemma2idx.items()}

        self.tag2idx = {tag: idx for idx, tag in enumerate(self.tag_encoder.classes_)}
        self.idx2tag = {idx: tag for tag, idx in self.tag2idx.items()}

    def split_dataset(self, test_size=0.2, random_state=42):
        X_train, X_test, Y_train, Y_test = train_test_split(
            self.X, self.Y, test_size=test_size, random_state=random_state, stratify=self.Y
        )
        return X_train, X_test, Y_train, Y_test

    def _train_and_evaluate(self, model, X_train, y_train, X_test, y_test):
        start_time = time.time()
        model.fit(X_train, y_train)
        training_time = time.time() - start_time

        y_pred = model.predict(X_test)

        token_f1 = f1_score(y_test, y_pred, average='micro')

        y_true_tags = [self.idx2tag[idx] for idx in y_test]
        y_pred_tags = [self.idx2tag[idx] for idx in y_pred]

        entity_f1 = entity_f1_score([y_true_tags], [y_pred_tags])

        self.model = model

        print(f"Finished Training: {self.model_name}")

        return {
            'Model': self.model_name,
            'Token F1': token_f1,
            'Entity F1': entity_f1,
            'Training Time': training_time
        }

    def train_naive_bayes(self, X_train, y_train, X_test, y_test):
        return self._train_and_evaluate(MultinomialNB(), X_train, y_train, X_test, y_test)


    def train_random_forest(self, X_train, y_train, X_test, y_test):
        return self._train_and_evaluate(
            RandomForestClassifier(n_jobs=-1),
            X_train, y_train, X_test, y_test
        )

    def train_extra_trees(self, X_train, y_train, X_test, y_test):
        return self._train_and_evaluate(
            ExtraTreesClassifier(n_jobs=-1),
            X_train, y_train, X_test, y_test
        )

    def train_gradient_boosting(self, X_train, y_train, X_test, y_test):
        return self._train_and_evaluate(
            GradientBoostingClassifier(),
            X_train, y_train, X_test, y_test
        )

    def train_xgboost(self, X_train, y_train, X_test, y_test):
        return self._train_and_evaluate(
            xgb.XGBClassifier(n_estimators=100, eval_metric='mlogloss'),
            X_train, y_train, X_test, y_test
        )

def naive_bayes_f(model_name, dataset):
    model = MachineLearningAlgorithmsNERFeatures(model_name, dataset)
    model.get_word_and_tags_2_ids()
    X_train, X_test, Y_train, Y_test = model.split_dataset()
    result = model.train_naive_bayes(X_train, Y_train, X_test, Y_test)
    return result

def extra_trees_f(model_name, dataset):
    model = MachineLearningAlgorithmsNERFeatures(model_name, dataset)
    model.get_word_and_tags_2_ids()
    X_train, X_test, Y_train, Y_test = model.split_dataset()
    result = model.train_extra_trees(X_train, Y_train, X_test, Y_test)
    return result

def random_forest_f(model_name, dataset):
    model = MachineLearningAlgorithmsNERFeatures(model_name, dataset)
    model.get_word_and_tags_2_ids()
    X_train, X_test, Y_train, Y_test = model.split_dataset()
    result = model.train_random_forest(X_train, Y_train, X_test, Y_test)
    return result

def xgboost_f(model_name, dataset):
    model = MachineLearningAlgorithmsNERFeatures(model_name, dataset)
    model.get_word_and_tags_2_ids()
    X_train, X_test, Y_train, Y_test = model.split_dataset()
    result = model.train_xgboost(X_train, Y_train, X_test, Y_test)
    return result

def gradient_boosting_f(model_name, dataset):
    model = MachineLearningAlgorithmsNERFeatures(model_name, dataset)
    model.get_word_and_tags_2_ids()
    X_train, X_test, Y_train, Y_test = model.split_dataset()
    result = model.train_gradient_boosting(X_train, Y_train, X_test, Y_test)
    return result



In [ ]:
class MLPNER:
    def __init__(self,dataset,model_name):
        self.dataset = dataset
        self.model_name = model_name
        self.word2index = {}
        self.tag2index = {}
        self.index2word = {}
        self.index2tag = {}

        self.X = None
        self.Y = None
        self.sentence_ids = None
        self.X_train, self.X_test, self.Y_train, self.Y_test = None, None, None, None
        self.sent_ids_train, self.sent_ids_test = None, None

    def build_vocab(self):
        all_words = self.dataset["WORD"].tolist()
        all_tags = self.dataset["NER_TAG"].tolist()

        unique_words = list(set(all_words))
        unique_tags = list(set(all_tags))

        self.word2index = {word: idx for idx, word in enumerate(unique_words)}
        self.index2word = {idx: word for word, idx in self.word2index.items()}

        self.tag2index = {tag: idx for idx, tag in enumerate(unique_tags)}
        self.index2tag = {idx: tag for tag, idx in self.tag2index.items()}

    def encode_dataset(self):
        words = self.dataset["WORD"].map(self.word2index).tolist()
        tags = self.dataset["NER_TAG"].map(self.tag2index).tolist()
        sentence_ids = self.dataset["SENTENCE #"].tolist()

        self.X = np.array(words).reshape(-1, 1)
        self.Y = np.array(tags)
        self.sentence_ids = np.array(sentence_ids)

    def split_dataset(self, test_size=0.2, random_state=None):

        self.X_train, self.X_test, self.Y_train, self.Y_test, self.sent_ids_train, self.sent_ids_test = train_test_split(
            self.X, self.Y, self.sentence_ids, test_size=test_size, random_state=random_state
        )

    def convert_to_seqeval_format(self, y_true, y_pred, sentence_ids):
        true_tags = [self.index2tag[idx] for idx in y_true]
        pred_tags = [self.index2tag[idx] for idx in y_pred]

        sentences_true = {}
        sentences_pred = {}

        for i, sent_id in enumerate(sentence_ids):
            if sent_id not in sentences_true:
                sentences_true[sent_id] = []
                sentences_pred[sent_id] = []
            sentences_true[sent_id].append(true_tags[i])
            sentences_pred[sent_id].append(pred_tags[i])

        y_true_seqeval = [sentences_true[sent_id] for sent_id in sorted(sentences_true.keys())]
        y_pred_seqeval = [sentences_pred[sent_id] for sent_id in sorted(sentences_pred.keys())]

        return y_true_seqeval, y_pred_seqeval

    def train(self, classifier="mlps"):

        classifiers = {
            "mlps": MLPClassifier(hidden_layer_sizes=(40,), max_iter=250, activation="relu", solver="adam"),
            "mlpm": MLPClassifier(hidden_layer_sizes=(120, 40), max_iter=250, activation="relu", solver="adam"),
            "mlpl": MLPClassifier(hidden_layer_sizes=(100, 120, 40), max_iter=250, activation="relu", solver="adam"),
        }

        model = classifiers[classifier]

        start_time = time.time()
        model.fit(self.X_train, self.Y_train)
        training_time = time.time() - start_time

        y_pred = model.predict(self.X_test)

        token_f1 = f1_score(self.Y_test, y_pred, average="micro")

        y_true_seqeval, y_pred_seqeval = self.convert_to_seqeval_format(self.Y_test, y_pred, self.sent_ids_test)
        entity_f1 = entity_f1_score(y_true_seqeval, y_pred_seqeval)

        print(f"Finished Training: {self.model_name}")

        return {
            'Model': self.model_name,
            'Token F1': token_f1,
            'Entity F1': entity_f1,
            'Training Time': training_time
        }

def train_MLPS(dataset,model_name):
    mlp = MLPNER(dataset,model_name)
    mlp.build_vocab()
    mlp.encode_dataset()
    mlp.split_dataset()
    results = mlp.train(classifier="mlps")
    return results

def train_MLPM(dataset,model_name):
    mlp = MLPNER(dataset,model_name)
    mlp.build_vocab()
    mlp.encode_dataset()
    mlp.split_dataset()
    results = mlp.train(classifier="mlpm")
    return results

def train_MLPL(dataset,model_name):
    mlp = MLPNER(dataset,model_name)
    mlp.build_vocab()
    mlp.encode_dataset()
    mlp.split_dataset()
    results = mlp.train(classifier="mlpl")
    return results

In [ ]:
class MLPNERFeatures:
    def __init__(self, dataset, model_name):
        self.dataset = dataset
        self.model_name = model_name
        self.word2index = {}
        self.pos2index = {}
        self.lemma2index = {}
        self.tag2index = {}
        self.index2word = {}
        self.index2pos = {}
        self.index2lemma = {}
        self.index2tag = {}

        self.X = None
        self.Y = None
        self.sentence_ids = None
        self.X_train, self.X_test, self.Y_train, self.Y_test = None, None, None, None
        self.sent_ids_train, self.sent_ids_test = None, None

    def build_vocab(self):
        all_words = self.dataset["WORD"].tolist()
        all_pos = self.dataset["POS_TAG"].tolist()
        all_lemmas = self.dataset["LEMMA"].tolist()
        all_tags = self.dataset["NER_TAG"].tolist()

        unique_words = list(set(all_words))
        unique_pos = list(set(all_pos))
        unique_lemmas = list(set(all_lemmas))
        unique_tags = list(set(all_tags))

        self.word2index = {word: idx for idx, word in enumerate(unique_words)}
        self.index2word = {idx: word for word, idx in self.word2index.items()}

        self.pos2index = {pos: idx for idx, pos in enumerate(unique_pos)}
        self.index2pos = {idx: pos for pos, idx in self.pos2index.items()}

        self.lemma2index = {lemma: idx for idx, lemma in enumerate(unique_lemmas)}
        self.index2lemma = {idx: lemma for lemma, idx in self.lemma2index.items()}

        self.tag2index = {tag: idx for idx, tag in enumerate(unique_tags)}
        self.index2tag = {idx: tag for tag, idx in self.tag2index.items()}

    def encode_dataset(self):
        words = self.dataset["WORD"].map(self.word2index).tolist()
        pos_tags = self.dataset["POS_TAG"].map(self.pos2index).tolist()
        lemmas = self.dataset["LEMMA"].map(self.lemma2index).tolist()
        tags = self.dataset["NER_TAG"].map(self.tag2index).tolist()
        sentence_ids = self.dataset["SENTENCE #"].tolist()

        features = np.column_stack((words, pos_tags, lemmas))

        self.X = features
        self.Y = np.array(tags)
        self.sentence_ids = np.array(sentence_ids)

    def split_dataset(self, test_size=0.2, random_state=None):
        self.X_train, self.X_test, self.Y_train, self.Y_test, self.sent_ids_train, self.sent_ids_test = train_test_split(
            self.X, self.Y, self.sentence_ids, test_size=test_size, random_state=random_state
        )

    def convert_to_seqeval_format(self, y_true, y_pred, sentence_ids):
        true_tags = [self.index2tag[idx] for idx in y_true]
        pred_tags = [self.index2tag[idx] for idx in y_pred]

        sentences_true = {}
        sentences_pred = {}

        for i, sent_id in enumerate(sentence_ids):
            if sent_id not in sentences_true:
                sentences_true[sent_id] = []
                sentences_pred[sent_id] = []
            sentences_true[sent_id].append(true_tags[i])
            sentences_pred[sent_id].append(pred_tags[i])

        y_true_seqeval = [sentences_true[sent_id] for sent_id in sorted(sentences_true.keys())]
        y_pred_seqeval = [sentences_pred[sent_id] for sent_id in sorted(sentences_pred.keys())]

        return y_true_seqeval, y_pred_seqeval

    def train(self, classifier="mlps"):
        classifiers = {
            "mlps": MLPClassifier(hidden_layer_sizes=(40,), max_iter=250, activation="relu", solver="adam"),
            "mlpm": MLPClassifier(hidden_layer_sizes=(120, 40), max_iter=250, activation="relu", solver="adam"),
            "mlpl": MLPClassifier(hidden_layer_sizes=(100, 120, 40), max_iter=250, activation="relu", solver="adam"),
        }

        model = classifiers[classifier]

        start_time = time.time()
        model.fit(self.X_train, self.Y_train)
        training_time = time.time() - start_time

        y_pred = model.predict(self.X_test)

        token_f1 = f1_score(self.Y_test, y_pred, average="micro")

        y_true_seqeval, y_pred_seqeval = self.convert_to_seqeval_format(self.Y_test, y_pred, self.sent_ids_test)
        entity_f1 = entity_f1_score(y_true_seqeval, y_pred_seqeval)

        print(f"Finished Training: {self.model_name}")

        return {
            'Model':self.model_name,
            'Token F1': token_f1,
            'Entity F1': entity_f1,
            'Training Time': training_time
        }


def train_MLPS_f(dataset,model_name):
    mlp = MLPNERFeatures(dataset,model_name)
    mlp.build_vocab()
    mlp.encode_dataset()
    mlp.split_dataset()
    results = mlp.train(classifier="mlps")
    return results


def train_MLPM_f(dataset,model_name):
    mlp = MLPNERFeatures(dataset,model_name)
    mlp.build_vocab()
    mlp.encode_dataset()
    mlp.split_dataset()
    results = mlp.train(classifier="mlpm")
    return results


def train_MLPL_f(dataset,model_name):
    mlp = MLPNERFeatures(dataset,model_name)
    mlp.build_vocab()
    mlp.encode_dataset()
    mlp.split_dataset()
    results = mlp.train(classifier="mlpl")
    return results

In [ ]:
naive = naive_bayes("Naive Bayes",dataset)
naive_f = naive_bayes_f("Naive Bayes (POS+LEMMA)",dataset)


extra = extra_trees("Extra Trees", dataset)
extra_f = extra_trees_f("Extra Trees (POS+LEMMA)", dataset)

forest = random_forest("Random Forest", dataset)
forest_f = random_forest_f("Random Forest (POS+LEMMA)", dataset)

xgboost = xgboost("XGBoost", dataset)
xgboost_f = xgboost_f("XGBoost (POS+LEMMA)", dataset)

gradient = gradient_boosting("Gradient Boosting", dataset)
gradient_f = gradient_boosting_f("Gradient Boosting (POS+LEMMA)", dataset)

mlps = train_MLPS(dataset,"MLPS")
mlps_f = train_MLPS_f(dataset,"MLPS (POS+LEMMA)")

mlpm = train_MLPM(dataset,"MLPM")
mlpm_f = train_MLPM_f(dataset,"MLPM (POS+LEMMA)")

mlpl = train_MLPL(dataset,"MLPL")
mlpl_f = train_MLPL_f(dataset,"MLPL (POS+LEMMA)")


Finished Training: Naive Bayes
Finished Training: Naive Bayes (POS+LEMMA)
Finished Training: Extra Trees
Finished Training: Extra Trees (POS+LEMMA)
Finished Training: Random Forest
Finished Training: Random Forest (POS+LEMMA)
Finished Training: XGBoost
Finished Training: XGBoost (POS+LEMMA)
Finished Training: Gradient Boosting
Finished Training: Gradient Boosting (POS+LEMMA)
Finished Training: MLPS
Finished Training: MLPS (POS+LEMMA)
Finished Training: MLPM
Finished Training: MLPM (POS+LEMMA)
Finished Training: MLPL
Finished Training: MLPL (POS+LEMMA)


In [ ]:
results = [naive,naive_f, extra,extra_f, forest,forest_f, xgboost, xgboost_f, gradient, gradient_f, mlps, mlps_f, mlpm, mlpm_f, mlpl, mlpl_f]
df_all = pd.DataFrame(results)

In [ ]:
df_all

,Model,Token F1,Entity F1,Training Time
0,Naive Bayes,0.909041,0.000000,0.353189
1,Naive Bayes (POS+LEMMA),0.669424,0.086729,0.880563
2,Extra Trees,0.961089,0.800389,39.043109
3,Extra Trees (POS+LEMMA),0.961612,0.803670,34.574355
4,Random Forest,0.961157,0.799843,118.726556
5,Random Forest (POS+LEMMA),0.961514,0.801727,84.104087
6,XGBoost,0.916408,0.261521,141.894877
7,XGBoost (POS+LEMMA),0.925177,0.429927,149.730034
8,Gradient Boosting,0.759894,0.177133,1380.912870
9,Gradient Boosting (POS+LEMMA),0.664216,0.220943,1508.865165


In [ ]:
print(df_all.to_string(index=False))

                        Model  Token F1  Entity F1  Training Time
                  Naive Bayes  0.909041   0.000000       0.353189
      Naive Bayes (POS+LEMMA)  0.669424   0.086729       0.880563
                  Extra Trees  0.961089   0.800389      39.043109
      Extra Trees (POS+LEMMA)  0.961612   0.803670      34.574355
                Random Forest  0.961157   0.799843     118.726556
    Random Forest (POS+LEMMA)  0.961514   0.801727      84.104087
                      XGBoost  0.916408   0.261521     141.894877
          XGBoost (POS+LEMMA)  0.925177   0.429927     149.730034
            Gradient Boosting  0.759894   0.177133    1380.912870
Gradient Boosting (POS+LEMMA)  0.664216   0.220943    1508.865165
                         MLPS  0.907590   0.000000     219.056371
             MLPS (POS+LEMMA)  0.908704   0.000334     108.817511
                         MLPM  0.908748   0.000000     434.677066
             MLPM (POS+LEMMA)  0.908631   0.000670     256.552266
          